# Compléments d'IA - TD2
## CSP

## ARSENA Claire GA1

### Exercice 1

Modélisation du Sudoku en CSP avec le formalisme vu en cours (X, D, C, R) :

X = une variable par case de la grille, donc 81 variables au total

D = pour une case déjà remplie le domaine est juste la valeur imposée, pour une case vide le domaine est 1 à 9

C = une contrainte "toutes différentes" par ligne, une par colonne, une par bloc 3x3

R = pour chaque contrainte, les tuples autorisés sont ceux où les 9 valeurs sont toutes distinctes (donc les permutations de 1 à 9)

Une solution c'est une affectation complète qui respecte toutes ces contraintes, càd aucune ligne colonne ou bloc avec un chiffre en double.

### Exercice 2

Grille représentée en liste de listes, 0 = case vide. On écrit Backtrack comme dans l'algo du cours : on prend une case vide, on essaie les valeurs une par une, si ça viole une contrainte on passe à la valeur suivante, sinon on continue récursivement, et si ça bloque on revient en arrière.

In [1]:
import copy
import time

grille_initiale = [
    [4, 0, 0, 3, 0, 0, 7, 6, 0],
    [0, 9, 3, 0, 7, 0, 4, 0, 1],
    [2, 0, 0, 1, 4, 0, 0, 8, 3],
    [9, 5, 8, 0, 0, 0, 0, 0, 7],
    [3, 4, 0, 0, 0, 7, 0, 1, 0],
    [0, 7, 2, 8, 9, 3, 5, 4, 0],
    [0, 8, 0, 2, 0, 0, 3, 7, 4],
    [0, 0, 4, 0, 0, 0, 1, 9, 5],
    [0, 0, 0, 0, 5, 0, 6, 0, 0],
]

def afficher_grille(g):
    for i, row in enumerate(g):
        if i % 3 == 0 and i != 0:
            print("-" * 21)
        ligne = ""
        for j, val in enumerate(row):
            if j % 3 == 0 and j != 0:
                ligne += "| "
            ligne += (str(val) if val != 0 else ".") + " "
        print(ligne)

print("Grille initiale :")
afficher_grille(grille_initiale)

Grille initiale :
4 . . | 3 . . | 7 6 . 
. 9 3 | . 7 . | 4 . 1 
2 . . | 1 4 . | . 8 3 
---------------------
9 5 8 | . . . | . . 7 
3 4 . | . . 7 | . 1 . 
. 7 2 | 8 9 3 | 5 4 . 
---------------------
. 8 . | 2 . . | 3 7 4 
. . 4 | . . . | 1 9 5 
. . . | . 5 . | 6 . . 


In [2]:
def est_consistant(grille, ligne, col, val):
    # verifie ligne, colonne et bloc 3x3
    for j in range(9):
        if grille[ligne][j] == val:
            return False
    for i in range(9):
        if grille[i][col] == val:
            return False
    bl, bc = 3 * (ligne // 3), 3 * (col // 3)
    for i in range(bl, bl + 3):
        for j in range(bc, bc + 3):
            if grille[i][j] == val:
                return False
    return True

def case_vide(grille):
    # renvoie la premiere case vide trouvee, None si grille pleine
    for i in range(9):
        for j in range(9):
            if grille[i][j] == 0:
                return (i, j)
    return None

def Backtrack(grille):
    case = case_vide(grille)
    if case is None:
        return True

    ligne, col = case
    for v in range(1, 10):
        if est_consistant(grille, ligne, col, v):
            grille[ligne][col] = v
            if Backtrack(grille):
                return True
            grille[ligne][col] = 0
    return False

In [3]:
grille_bt = copy.deepcopy(grille_initiale)

debut = time.time()
solution_trouvee = Backtrack(grille_bt)
duree = time.time() - debut

print("Solution trouvée :", solution_trouvee)
print("Temps de résolution : {:.4f} s".format(duree))
print()
afficher_grille(grille_bt)

Solution trouvée : True
Temps de résolution : 0.0007 s

4 1 5 | 3 8 9 | 7 6 2 
8 9 3 | 6 7 2 | 4 5 1 
2 6 7 | 1 4 5 | 9 8 3 
---------------------
9 5 8 | 4 1 6 | 2 3 7 
3 4 6 | 5 2 7 | 8 1 9 
1 7 2 | 8 9 3 | 5 4 6 
---------------------
5 8 9 | 2 6 1 | 3 7 4 
6 2 4 | 7 3 8 | 1 9 5 
7 3 1 | 9 5 4 | 6 2 8 


Ca resout la grille quasi instantanement. Des qu'une valeur viole une contrainte on passe direct a la suivante, donc on n'explore pas des branches qui de toute facon ne mèneraient a rien, contrairement au Generate and Test vu en cours qui teste bêtement toutes les affectations completes.

### Exercice 3

Ajout du Forward Checking : a chaque fois qu'on affecte une valeur, on l'enleve des domaines des cases voisines (meme ligne, meme colonne, meme bloc). Si un domaine se retrouve vide, on le sait tout de suite sans attendre d'essayer d'instancier cette case.

In [4]:
def voisins(ligne, col):
    # cases liees par une contrainte a (ligne, col)
    v = set()
    for j in range(9):
        if j != col:
            v.add((ligne, j))
    for i in range(9):
        if i != ligne:
            v.add((i, col))
    bl, bc = 3 * (ligne // 3), 3 * (col // 3)
    for i in range(bl, bl + 3):
        for j in range(bc, bc + 3):
            if (i, j) != (ligne, col):
                v.add((i, j))
    return v

def init_domaines(grille):
    domaines = {}
    for i in range(9):
        for j in range(9):
            if grille[i][j] == 0:
                domaines[(i, j)] = set(range(1, 10))
            else:
                domaines[(i, j)] = {grille[i][j]}
    for i in range(9):
        for j in range(9):
            if grille[i][j] != 0:
                for (pi, pj) in voisins(i, j):
                    domaines[(pi, pj)].discard(grille[i][j])
    return domaines

In [5]:
def Forward_Checking(grille, domaines, cases_libres):
    if not cases_libres:
        return True

    x = cases_libres[0]  # ordre naturel, pas d'heuristique dans cet exo
    ligne, col = x
    reste = cases_libres[1:]

    for v in sorted(domaines[x]):
        sauvegarde = []
        consistant = True
        grille[ligne][col] = v

        for xj in voisins(ligne, col):
            if xj in domaines and v in domaines[xj]:
                domaines[xj].discard(v)
                sauvegarde.append(xj)
                if len(domaines[xj]) == 0 and grille[xj[0]][xj[1]] == 0:
                    consistant = False

        if consistant and Forward_Checking(grille, domaines, reste):
            return True

        for xj in sauvegarde:
            domaines[xj].add(v)
        grille[ligne][col] = 0

    return False

In [6]:
grille_fc = copy.deepcopy(grille_initiale)
domaines_fc = init_domaines(grille_fc)
cases_libres = [(i, j) for i in range(9) for j in range(9) if grille_fc[i][j] == 0]

debut = time.time()
solution_trouvee = Forward_Checking(grille_fc, domaines_fc, cases_libres)
duree = time.time() - debut

print("Solution trouvée :", solution_trouvee)
print("Temps de résolution : {:.4f} s".format(duree))
print()
afficher_grille(grille_fc)

Solution trouvée : True
Temps de résolution : 0.0005 s

4 1 5 | 3 8 9 | 7 6 2 
8 9 3 | 6 7 2 | 4 5 1 
2 6 7 | 1 4 5 | 9 8 3 
---------------------
9 5 8 | 4 1 6 | 2 3 7 
3 4 6 | 5 2 7 | 8 1 9 
1 7 2 | 8 9 3 | 5 4 6 
---------------------
5 8 9 | 2 6 1 | 3 7 4 
6 2 4 | 7 3 8 | 1 9 5 
7 3 1 | 9 5 4 | 6 2 8 


On voit que ca explore moins de noeuds que le backtracking simple. Les domaines vides sont detectes direct a la mise a jour, donc ca coupe des branches plus tot, avant meme d'essayer d'instancier la case concernee.

### Exercice 4

L'ordre des valeurs n'a pas d'impact sur un probleme inconsistant, parce qu'un probleme inconsistant n'a aucune solution. Peu importe l'ordre dans lequel on teste les valeurs, l'algo va toutes les essayer et toutes echouer avant de remonter. Ca change juste l'ordre dans lequel on explore les branches, mais elles echouent toutes de toute facon donc l'espace parcouru reste le meme au final. C'est l'ordre des variables qui peut vraiment changer les choses, en permettant de detecter l'echec plus vite (principe de l'echec d'abord).

### Exercice 5

Modification du Forward Checking pour choisir a chaque etape la case avec le plus petit domaine restant (heuristique MRV), au lieu de prendre les cases dans l'ordre.

In [7]:
def Forward_Checking_MRV(grille, domaines, cases_libres):
    if not cases_libres:
        return True

    # on prend la case avec le domaine le plus petit
    x = min(cases_libres, key=lambda c: len(domaines[c]))
    ligne, col = x
    reste = [c for c in cases_libres if c != x]

    for v in sorted(domaines[x]):
        sauvegarde = []
        consistant = True
        grille[ligne][col] = v

        for xj in voisins(ligne, col):
            if xj in domaines and v in domaines[xj]:
                domaines[xj].discard(v)
                sauvegarde.append(xj)
                if len(domaines[xj]) == 0 and grille[xj[0]][xj[1]] == 0:
                    consistant = False

        if consistant and Forward_Checking_MRV(grille, domaines, reste):
            return True

        for xj in sauvegarde:
            domaines[xj].add(v)
        grille[ligne][col] = 0

    return False

In [8]:
grille_mrv = copy.deepcopy(grille_initiale)
domaines_mrv = init_domaines(grille_mrv)
cases_libres_mrv = [(i, j) for i in range(9) for j in range(9) if grille_mrv[i][j] == 0]

debut = time.time()
solution_trouvee = Forward_Checking_MRV(grille_mrv, domaines_mrv, cases_libres_mrv)
duree = time.time() - debut

print("Solution trouvée :", solution_trouvee)
print("Temps de résolution : {:.4f} s".format(duree))
print()
afficher_grille(grille_mrv)

Solution trouvée : True
Temps de résolution : 0.0006 s

4 1 5 | 3 8 9 | 7 6 2 
8 9 3 | 6 7 2 | 4 5 1 
2 6 7 | 1 4 5 | 9 8 3 
---------------------
9 5 8 | 4 1 6 | 2 3 7 
3 4 6 | 5 2 7 | 8 1 9 
1 7 2 | 8 9 3 | 5 4 6 
---------------------
5 8 9 | 2 6 1 | 3 7 4 
6 2 4 | 7 3 8 | 1 9 5 
7 3 1 | 9 5 4 | 6 2 8 


In [9]:
# comparaison du nombre de noeuds explores par les 3 versions

def Backtrack_compte(grille, compteur):
    compteur[0] += 1
    case = case_vide(grille)
    if case is None:
        return True
    ligne, col = case
    for v in range(1, 10):
        if est_consistant(grille, ligne, col, v):
            grille[ligne][col] = v
            if Backtrack_compte(grille, compteur):
                return True
            grille[ligne][col] = 0
    return False

def Forward_Checking_compte(grille, domaines, cases_libres, compteur, mrv=False):
    compteur[0] += 1
    if not cases_libres:
        return True
    if mrv:
        x = min(cases_libres, key=lambda c: len(domaines[c]))
        reste = [c for c in cases_libres if c != x]
    else:
        x = cases_libres[0]
        reste = cases_libres[1:]
    ligne, col = x
    for v in sorted(domaines[x]):
        sauvegarde = []
        consistant = True
        grille[ligne][col] = v
        for xj in voisins(ligne, col):
            if xj in domaines and v in domaines[xj]:
                domaines[xj].discard(v)
                sauvegarde.append(xj)
                if len(domaines[xj]) == 0 and grille[xj[0]][xj[1]] == 0:
                    consistant = False
        if consistant and Forward_Checking_compte(grille, domaines, reste, compteur, mrv):
            return True
        for xj in sauvegarde:
            domaines[xj].add(v)
        grille[ligne][col] = 0
    return False

g1 = copy.deepcopy(grille_initiale)
c1 = [0]
Backtrack_compte(g1, c1)

g2 = copy.deepcopy(grille_initiale)
d2 = init_domaines(g2)
cl2 = [(i, j) for i in range(9) for j in range(9) if g2[i][j] == 0]
c2 = [0]
Forward_Checking_compte(g2, d2, cl2, c2, mrv=False)

g3 = copy.deepcopy(grille_initiale)
d3 = init_domaines(g3)
cl3 = [(i, j) for i in range(9) for j in range(9) if g3[i][j] == 0]
c3 = [0]
Forward_Checking_compte(g3, d3, cl3, c3, mrv=True)

print("Nombre de noeuds explores :")
print("  Backtracking simple      :", c1[0])
print("  Forward Checking         :", c2[0])
print("  Forward Checking + MRV   :", c3[0])

Nombre de noeuds explores :
  Backtracking simple      : 86
  Forward Checking         : 45
  Forward Checking + MRV   : 42


On voit que le nombre de noeuds diminue a chaque fois. Le Forward Checking detecte les echecs plus tot grace aux domaines, et le MRV coupe encore plus vite en instanciant en priorite les cases les plus contraintes. Ici l'ecart est pas enorme parce que la grille est deja assez facile, mais sur une grille plus dure ou moins remplie l'effet serait beaucoup plus visible.